[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AGRIdigital-Climbat/PlantillaMapa/blob/main/12_Sintesis/01_sintesis.ipynb)

# Sintesis: matriz de indicadores de una fila por municipio
## PROSUCO - CLIMBAT | Analisis por municipio de Bolivia (GEE + Colab)

> **Area de estudio = tu GeoJSON.** Este notebook usa por defecto:
>
> `RUTA_AOI = "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"` (compartido `Cualquiera con el enlace`).
> El GeoJSON **siempre manda**: al cargarse, `roi`, `ZONA['bbox']`, `ZONA['centro']`
> y `area_km2` se recalculan desde el poligono. El `bbox` de CONFIG solo es respaldo
> si cambias a `AOI_MODE = "bbox"`.

**Orden de ejecucion:** 1) Instalacion -> 2) Autenticacion GEE (`EE_PROJECT`) ->
3) CONFIG (anos + satelite + umbrales) -> 4) AOI desde Drive -> 5) celdas del modulo.
Solo edita la celda **CONFIGURACION MAESTRA**.


In [ ]:
# ============================================================
# 1.1 Instalación de librerías (ejecutar una sola vez en Colab)
# ============================================================

import importlib
import subprocess
import sys


def _asegurar(paquete, modulo=None):
    """Instala el paquete solo si no está disponible."""
    try:
        importlib.import_module(modulo or paquete)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", paquete]
        )


_asegurar("earthengine-api", "ee")
_asegurar("geemap")
_asegurar("geopandas")
_asegurar("pandas")
_asegurar("numpy")
_asegurar("matplotlib")

import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

print("Librerías listas.")
print("earthengine-api:", ee.__version__)
print("geemap:", geemap.__version__)

In [ ]:
# ============================================================
# 1.2 Autenticación en Google Earth Engine
# ============================================================
# >>> REEMPLAZAR por el id del proyecto de Cloud de PROSUCO/CLIMBAT <<<

EE_PROJECT = "iotapp7"

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

print("Earth Engine inicializado correctamente.")
print("Proyecto:", EE_PROJECT)

## Configuración maestra

> **✏️ Esta es la única celda que se edita para diagnosticar una zona nueva.**

| Bloque | Qué define | Verificación previa |
|---|---|---|
| `ZONA` | nombre, bbox `[oeste, sur, este, norte]` y punto de vista | dibujar en [geojson.io](https://geojson.io) o en el editor GEE y copiar coordenadas |
| `AOI_MODE`/`RUTA_AOI` | fuente del AOI: `"bbox"` o `"auto"` (polígono **GeoJSON**) | exportar el polígono desde QGIS como **GeoJSON**; en `RUTA_AOI` se puede pegar un **enlace de Google Drive** (el notebook lo descarga solo) o una ruta de Drive |
| `PUNTOS_LOCALES` | manantiales/bofedales del saber local | levantamiento con las comunidades |
| `FECHA_INICIO / FECHA_FIN` | ventana analítica (**30+ años**) | default 1996-01-01 → 2026-12-31 |
| `CALENDARIO` | meses de siembra/cosecha y estación seca | observadores Yapuchiri |
| `UMBRALES` | todos los cortes analíticos en un solo lugar | criterio técnico + experiencia local |
| `ESCALAS` | resolución de cómputo (m); subir si la zona es grande | — |

In [ ]:
# ===========================================================================
# CONFIGURACIÓN MAESTRA — ÚNICA CELDA QUE SE EDITA PARA CADA ZONA
# ===========================================================================

# --- 1. Zona de intervención ------------------------------------------------
# Fuente del AOI:
#   "bbox" -> usa el rectángulo ZONA["bbox"] de abajo (rápido, sin Drive)
#   "auto" -> usa un polígono .geojson. RUTA_AOI admite 3 formas:
#       * Enlace o ID de Google Drive  -> se DESCARGA solo (no subes nada a mano)
#           "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"
#           "11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N"
#       * Ruta en tu Drive ya montado  -> se lee directo, SIN copiar a /content
#           "/content/drive/MyDrive/CLIMBAT/zona.geojson"
#       * Carpeta en Drive             -> toma el único .geojson que contenga
#           "/content/drive/MyDrive/CLIMBAT"
AOI_MODE = "auto"
RUTA_AOI = "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"

ZONA = {
    # Coordenadas geocodificadas: Escoma (La Paz, Bolivia) ~3832 msnm
    # (solo se usan como respaldo cuando AOI_MODE = "bbox")
    "nombre": "Escoma, La Paz (prueba)",
    "bbox": [-69.33, -15.86, -68.93, -15.46],   # [oeste, sur, este, norte]
    "centro": [-69.1289, -15.6615],             # punto de vista del mapa
    "zoom": 10,
}

# --- 2. Puntos levantados con comunidades (saber local) ---------------------
# Listas de [longitud, latitud]. Dejar vacías si aún no hay levantamiento.
PUNTOS_LOCALES = {
    "manantiales": [],
    "bofedales": [],
}

# --- 3. Ventana analítica: 30+ años hacia atrás desde el año actual ---------
# Regla de la plantilla: 1996-01-01 -> 2026-12-31 (todo análisis de rango de años usa esta).
# Los datasets usan su período nativo si es mayor: CHIRPS 1981+ · ERA5-Land 1950+
# · JRC 1984–2021 · MODIS 2000+ (NDVI) / 2001+ (MCD12Q1 agricultura) · SMAP 2015+
FECHA_INICIO = "1996-01-01"
FECHA_FIN = "2026-12-31"

# --- 4. Calendario agrícola local (verificar con observadores Yapuchiri) ----
CALENDARIO = {
    "siembra": 11,       # mes de siembra (1–12)
    "cosecha": 4,        # mes de cosecha (1–12)
    "meses_estacion_seca": [5, 6, 7, 8, 9, 10],   # se usa en M10 (bofedales)
}

# --- 5. Umbrales analíticos (ajustar con criterio técnico y local) ----------
UMBRALES = {
    "helada_c": 0.0,               # °C mínima para contar un día de helada (M4)
    "helada_grave_c": -2.0,        # °C de helada grave (daño severo) (M4)
    "mes_seco_mm": 30.0,           # mm/mes para clasificar mes seco (M2)
    "spi_severo": -1.5,            # umbral de sequía severa SPI-3 (M5)
    "ndvi_anomalia_z": -1.5,       # anomalía NDVI severa (M7)
    "manantial_occurrencia": 100,  # % ocurrencia JRC mínima (M9)
    "manantial_drenaje_max": 50,   # celdas acumuladas máx. (cabeceras) (M9)
    "bofedal_altitud_min": 3000,   # msnm mínimo (M10)
    "bofedal_altitud_max": 5200,   # msnm máximo (M10)
    "bofedal_pendiente_max": 5.0,  # grados de pendiente máxima (M10)
    "bofedal_seasonality": 8,      # meses con agua JRC mínimos (M10)
    "bofedal_ndvi_seco": 0.25,     # NDVI mínimo en estación seca (M10)
    "clases_agricolas": [12, 14],  # LC_Type1 MCD12Q1: 12=cultivos, 14=mosaico (M11)
}

# --- 6. Escalas de cómputo (metros) -----------------------------------------
# Subir si la zona es muy grande (más rápido, menos detalle)
ESCALAS = {"clima": 5000, "modis": 250, "jrc": 90, "detalle": 30}
TILE_SCALE = 4

# ===========================================================================
# A partir de aquí no se edita nada
# ===========================================================================

MESES = list(range(1, 13))
MESES_ES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
            "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

roi = None   # se construye en la celda 1.4 (bbox o GeoJSON según AOI_MODE)
centro = None
ANIO_INI = int(FECHA_INICIO[:4])
ANIO_FIN = int(FECHA_FIN[:4])

# Años COMPLETOS (se excluye el año en curso de métricas y mapas anuales)
ANIOS_COMPLETOS = list(range(ANIO_INI, ANIO_FIN))
FILTRO_ANIOS_COMPLETOS = ee.Filter.calendarRange(
    ANIOS_COMPLETOS[0], ANIOS_COMPLETOS[-1], "year"
)

print("Zona:", ZONA["nombre"])
print("Ventana analítica:", FECHA_INICIO, "->", FECHA_FIN,
      f"({ANIO_FIN - ANIO_INI + 1} años calendario)")
print("Años completos para métricas:", ANIOS_COMPLETOS[0], "-", ANIOS_COMPLETOS[-1])
print("Calendario: siembra mes", CALENDARIO["siembra"],
      "| cosecha mes", CALENDARIO["cosecha"])
print("Umbrales activos:", UMBRALES)

In [ ]:
# ============================================================
# 1.3 Utilidades comunes (reutilizadas por todos los módulos)
# ============================================================

# Parámetros de reducción seguros contra "User memory limit exceeded":
#   bestEffort=True -> GEE sube la escala automáticamente si no cabe en memoria.
#   tileScale=8     -> divide el cálculo en tiles más pequeños (menos memoria c/u).
REDUCIR = dict(maxPixels=1e9, tileScale=8, bestEffort=True)
REDUCIR_AREA = dict(maxPixels=1e10, tileScale=16, bestEffort=True)


def apilar(imagenes, nombres):
    """Une imágenes de una banda en una sola imagen multibanda."""
    pila = None
    for img, nombre in zip(imagenes, nombres):
        banda = ee.Image(img).select(0).rename(nombre)
        pila = banda if pila is None else pila.addBands(banda)
    return pila


def reducir_bandas(imagenes, nombres, escala, tam=6, reductor=None, params=None):
    """reduceRegion de muchas bandas EN BLOQUES (una petición por bloque).

    Evita el error de memoria que ocurre al reducir de una sola vez una imagen
    con decenas de bandas (cada banda puede depender de cientos/miles de imágenes
    diarias). Devuelve un dict {nombre_banda: valor}.
    """
    reductor = reductor or ee.Reducer.mean()
    params = params or REDUCIR
    salida = {}
    for i in range(0, len(imagenes), tam):
        pila = apilar(imagenes[i:i + tam], nombres[i:i + tam])
        salida.update(
            pila.reduceRegion(reductor, roi, escala, **params).getInfo()
        )
    return salida


def composite_mensual(coleccion, anios, banda=None):
    """Reduce una colección diaria a UN compuesto por año-mes (una imagen c/u).

    Aligera ~30x la memoria de las climatologías de 30 años. `anios` debe listar
    solo años con dato (evita colecciones vacías). Con `banda`, deja una sola banda.
    """
    imgs = []
    for y in anios:
        for m in MESES:
            i0 = f"{y}-{m:02d}-01"
            i1 = (pd.Timestamp(i0) + pd.offsets.MonthBegin(1)).strftime("%Y-%m-%d")
            sub = coleccion.filterDate(i0, i1)
            if banda:
                sub = sub.select(banda)
            imgs.append(sub.mean().set("system:time_start", ee.Date(i0).millis()))
    return ee.ImageCollection(imgs)


def borde_aoi(color="FF0000", ancho=2):
    """Capa de SOLO el borde del AOI (relleno transparente).

    Evita que el relleno de color tape el mapa y meta ruido en la interpretación;
    deja el límite del área bien visible por encima de las demás capas.
    """
    return roi.style(color=color, fillColor="00000000", width=ancho)


def promediar_aoi(imagen, banda, escala):
    """Promedio de una banda dentro del AOI -> valor Python (o None)."""
    d = (
        imagen.select(banda)
        .reduceRegion(ee.Reducer.mean(), roi, escala, **REDUCIR)
        .getInfo()
    )
    return d.get(banda)


def serie_aoi(coleccion, banda, escala, bloque_anios=5):
    """Extrae el promedio del AOI por imagen, EN BLOQUES de años.

    Procesar la serie completa de una vez agota la memoria; por eso se trocea
    (por defecto cada 5 años) y se concatenan los resultados.
    """
    registros = []

    def _una_imagen(img):
        st = (
            img.select(banda)
            .reduceRegion(ee.Reducer.mean(), roi, escala, **REDUCIR)
        )
        return ee.Feature(None, {
            "date": ee.Date(img.get("system:time_start")).format("yyyy-MM-dd"),
            "valor": st.get(banda),
        })

    for y0 in range(ANIO_INI, ANIO_FIN + 1, bloque_anios):
        y1 = min(y0 + bloque_anios - 1, ANIO_FIN)
        sub = coleccion.filterDate(f"{y0}-01-01", f"{y1 + 1}-01-01")
        fc = ee.FeatureCollection(sub.map(_una_imagen)).getInfo()
        registros += [f["properties"] for f in fc["features"]]
    return registros


print("Utilidades: apilar(), reducir_bandas(), composite_mensual(), "
      "promediar_aoi(), serie_aoi()")

In [ ]:
# ============================================================
# 1.4 Área de estudio (AOI) y mapa base
# ============================================================
# Fuente del AOI (ver CONFIG):
#   AOI_MODE = "bbox" -> rectángulo ZONA["bbox"].
#   AOI_MODE = "auto" -> polígono desde un archivo .geojson en Google Drive.
#                        (En QGIS: Exportar -> Guardar entidades como -> GeoJSON,
#                         CRS EPSG:4326; subir a Drive y ajustar RUTA_AOI.)

def borde_aoi(color="FF0000", ancho=2):
    """Capa de SOLO el borde del AOI (relleno transparente)."""
    # ee.Geometry NO tiene .style() en la API de Python -> se envuelve en FeatureCollection
    fc = ee.FeatureCollection([ee.Feature(roi)])
    return fc.style(color=color, fillColor="00000000", width=ancho)


if AOI_MODE == "auto":
    from google.colab import drive
    drive.mount("/content/drive")

    import os
    import re
    import glob
    import geopandas as gpd

    ruta = str(RUTA_AOI).strip()

    # (a) Enlace o ID de Google Drive -> descarga automática (sin subir nada a mano).
    #     El archivo debe estar compartido como "Cualquiera con el enlace".
    if ruta.startswith("http") or ("/" not in ruta and len(ruta) >= 20):
        _asegurar("gdown")
        import gdown

        m = re.search(r"(?:/d/|id=)([-\w]{20,})", ruta)
        fid = m.group(1) if m else ruta
        ext = (
            ".zip" if ruta.lower().endswith(".zip") else
            ".kml" if ruta.lower().endswith(".kml") else
            ".gpkg" if ruta.lower().endswith(".gpkg") else
            ".geojson"
        )
        destino = "/content/zona_aoi" + ext
        print("Descargando AOI desde Google Drive...")
        gdown.download(id=fid, output=destino, quiet=False)
        ruta = destino
    # (b) Carpeta en Drive -> usar el único .geojson que contenga
    elif os.path.isdir(ruta):
        cands = sorted(glob.glob(os.path.join(ruta, "*.geojson")))
        if not cands:
            raise FileNotFoundError(f"No se encontró ningún .geojson en {ruta}")
        ruta = cands[0]
    # (c) Ruta de archivo en el Drive montado -> se lee directo (sin copiar)

    gdf = gpd.read_file(ruta).to_crs(4326)
    try:
        geometria = gdf.geometry.union_all()      # shapely >= 2 (disuelve features)
    except AttributeError:
        geometria = gdf.geometry.unary_union      # shapely < 2
    gdf = gpd.GeoDataFrame(geometry=[geometria], crs=4326)

    roi = geemap.geopandas_to_ee(gdf).geometry()

    minx, miny, maxx, maxy = geometria.bounds
    ZONA["bbox"] = [minx, miny, maxx, maxy]
    ZONA["centro"] = [(minx + maxx) / 2, (miny + maxy) / 2]
    ZONA["zoom"] = 10
    print("AOI cargado desde:", ruta)
else:
    roi = ee.Geometry.Rectangle(ZONA["bbox"], proj="EPSG:4326", geodesic=False)

centro = ee.Geometry.Point(ZONA["centro"])
area_km2 = roi.area(maxError=1e6).divide(1e6).getInfo()

Map_aoi = geemap.Map()
Map_aoi.centerObject(roi, ZONA["zoom"])
Map_aoi.add_basemap("HYBRID")
Map_aoi.addLayer(borde_aoi(), {}, f"AOI: {ZONA['nombre']}")

# Puntos del saber local (si se cargaron en CONFIG)
for tipo, color in (("manantiales", "cyan"), ("bofedales", "magenta")):
    coords = PUNTOS_LOCALES.get(tipo, [])
    if coords:
        Map_aoi.addLayer(
            ee.Geometry.MultiPoint(coords),
            {"color": color},
            f"{tipo.capitalize()} (saber local)",
        )

print(f"Zona: {ZONA['nombre']}")
print(f"Área del AOI: {area_km2:,.1f} km²")
print(f"Extensión (bbox): {ZONA['bbox']}")
print("Puntos de saber local cargados:",
      {k: len(v) for k, v in PUNTOS_LOCALES.items()})

Map_aoi

## Marco de términos: las 8 dimensiones del análisis temporal

> Estas dimensiones **etiquetan cada indicador** de la plantilla. No son sinónimos.

| Dimensión | Pregunta | Ejemplo concreto en la zona |
|---|---|---|
| **Ocurrencia** | ¿Ocurrió / está presente el estado? | % de observaciones con agua en el píxel; días con lluvia / días totales |
| **Frecuencia** | ¿Cuántas veces por unidad de tiempo? (conteo) | nº de años con sequía severa en 30; nº de días de helada por año |
| **Estacionalidad** | ¿En qué mes del ciclo anual se concentra? | nº de meses con agua (0–12); curva climatológica de lluvia |
| **Recurrencia** | ¿Con qué regularidad **entre años** regresa? | % de años en que el evento repite en el mismo mes |
| **Temporalidad** | ¿Cuándo calza con el calendario agrícola? | inicio real de lluvias vs. siembra; mes de pico NDVI vs. calendario |
| **Duración** | ¿Cuánto dura el evento? | máxima racha de meses con SPI ≤ −1,5; días seguidos bajo 0 °C |
| **Intensidad** | ¿Qué tan fuerte fue? | SPI mínimo, nº °C bajo cero, mm de déficit acumulado |
| **Tendencia** | ¿Cómo evoluciona en el tiempo? | pendiente de temperatura (°C/década), km² de superficie agrícola por década |

<details>
<summary><b>Corrección de uso (viene del MD guía) y errores a evitar</b> — clic para ver</summary>
<br>

* En JRC Global Surface Water, `occurrence` se traduce como **ocurrencia** (participación
  temporal, en %), **no** como "frecuencia"; la *frecuencia* es siempre un **conteo de eventos**.
* **Ocurrencia ≠ recurrencia:** una cosa es *qué % del tiempo hay agua*; otra, *qué % de años
  repite en el mismo mes*.
* No decidir con una sola imagen: los sistemas dinámicos (lagos, bofedales, cultivos) requieren
  **series temporales**; por eso la ventana mínima es de **30 años**.
* Al comunicar tendencias, indicar **siempre el período y el dataset**; un ajuste lineal simple
  no sustituye pruebas formales (Mann-Kendall).

</details>

## Inventario de datos satelitales: primarios y alternativas

Todos los identificadores están verificados contra el **catálogo de Earth Engine**.

| Variable | Primario (default) | Alternativas (comentar/descomentar) | Resolución | Período | Módulo |
|---|---|---|---|---|---|
| Precipitación | `UCSB-CHG/CHIRPS/DAILY` | `NASA/GPM_L3/IMERG_V06` · `ECMWF/ERA5_LAND/DAILY_AGGR` · `IDAHO_EPSCOR/TERRACLIMATE` (pr) | 0,05° (~5 km) | 1981–actual | M2, M5 |
| Temperatura del aire | `ECMWF/ERA5_LAND/DAILY_AGGR` (`temperature_2m_min/max`) | `IDAHO_EPSCOR/TERRACLIMATE` (`tmmx/tmmn`, escala **×0,1**) · `MODIS/061/MOD11A2` (LST, 1 km) | ~11 km | 1950–actual | M3, M4 |
| Heladas | Conteo de Tmin ≤ 0 °C (ERA5-Land diario) | LST nocturna `MODIS/061/MOD11A1` (1 km) | 11 km / 1 km | 1950+ / 2000+ | M4 |
| Sequía | SPI-3 calculado con CHIRPS | PDSI de TerraClimate (×0,01) | ~5 km | 1981+ | M5 |
| Humedad de suelo | `NASA/SMAP/SPL4SMGP/008` (`sm_rootzone`, `sm_rootzone_pctl`, `sm_surface_anomaly`) | `NASA/SMAP/SPL4SMGP/007` · `ECMWF/ERA5_LAND/DAILY_AGGR` (`volumetric_soil_water_layer_1..4`) | 11 km | 2015–actual | M6 |
| NDVI / vegetación | `MODIS/061/MOD13Q1` (NDVI ×0,0001) | `COPERNICUS/S2_SR_HARMONIZED` (10 m, 2017+) · `LANDSAT/LC08/C02/T1_L2` | 250 m, 16 días | 2000–actual | M7 |
| **Superficie agrícola (30 años)** | `MODIS/061/MCD12Q1` (`LC_Type1`; clases 12 cultivos, 14 mosaico) | `GOOGLE/DYNAMICWORLD` (clase `crops`, 2015+, 10 m) · `ESA/WorldCover/v200` (2020/21) | 500 m, anual | **2001–actual** | **M11** |
| Coberturas actuales | `ESA/WorldCover/v200/2021` (banda `Map`) | `ESA/WorldCover/v200/2020` · `GOOGLE/DYNAMICWORLD` | 10 m | 2020–2021 | M12 |
| Cuerpos de agua | `JRC/GSW1_4/GlobalSurfaceWater` + `YearlyHistory` | MNDWI Sentinel-2 (extender después de 2021) | 30 m | 1984–2021 | M8 |
| Manantiales | Proxy: ocurrencia JRC + `WWF/HydroSHEDS/15ACC` + relieve | Validación con puntos del saber local (CONFIG) | 30–450 m | 1984–2021 | M9 |
| Bofedales | Proxy: NDVI en estación seca + JRC seasonality + altitud/pendiente | `ESA/WorldCover` clase 90 · clasificación RF Sentinel-1+2 (pesada) | 30–250 m | 2000+ | M10 |
| Fisiografía | `USGS/SRTMGL1_003` | `COPERNICUS/DEM/GLO30` · `NASA/NASADEM_HGT/001` | 30 m | estático | M1 |
| Drenaje | `WWF/HydroSHEDS/15ACC` (banda `b1`) | `WWF/HydroSHEDS/30ACC` | ~450 m | estático | M1, M9 |

<details>
<summary><b>Notas técnicas verificadas (escala, unidades, versiones)</b> — clic para ver</summary>
<br>

* `ECMWF/ERA5_LAND/DAILY_AGGR` incluye `temperature_2m_min/max` (**Kelvin**: restar 273,15) y
  `total_precipitation_sum` (**metros**: ×1000 para mm).
* `NASA/SMAP/SPL4SMGP/007` está **deprecado**; el vigente es **`008`** (mismas bandas:
  `sm_surface`, `sm_rootzone`, `sm_rootzone_pctl`, `sm_surface_anomaly`).
* `IDAHO_EPSCOR/TERRACLIMATE` aplica **factor de escala** (`tmmx/tmmn` ×0,1; `pdsi` ×0,01; `soil` ×0,1).
* JRC GSW **v1.4** cubre 1984–2021 completo en GEE (no se puede extender con el producto);
  para agua **reciente** se usa Sentinel-2 MNDWI (documentado en M8).
* `MODIS/061/MCD12Q1` inicia en **2001**: la ventana 1996–2000 queda marcada como *sin dato*
  en M11 (no se rellena con estimaciones).

</details>

# M13. Matriz de síntesis: una fila por territorio con los 13 módulos
Consolida todo lo calculado en un solo vistazo tabular (ideal para comparar varias zonas:
ejecutar la plantilla por zona y concatenar las filas).

**Requiere haber ejecutado M1–M12.** Si falta algo, la celda lo avisa y completa con "—".

In [ ]:
# ============================================================
# M13.1 Construcción de la matriz de indicadores
# ============================================================


def _v(nombre, default="— (ejecutar módulo previo)"):
    """Valor de una variable calculada en módulos previos, o aviso si falta."""
    return globals()[nombre] if nombre in globals() else default


def _num(nombre, fn=None, default="—"):
    """Valor numérico con transformación opcional; default si falta."""
    if nombre not in globals():
        return default
    val = globals()[nombre]
    try:
        return round(fn(val), 1) if fn else round(float(val), 1)
    except Exception:
        return default


indicadores = {}

# --- Territorio y fisiografía (M1) --------------------------------------------
indicadores["Zona"] = ZONA["nombre"]
indicadores["Área zona (km²)"] = _num("area_km2", lambda v: v)
indicadores["Elevación mín–máx (msnm)"] = (
    f"{E_MIN:.0f}–{E_MAX:.0f}" if "E_MIN" in globals() else "—"
)
indicadores["Pendiente media (°)"] = _num("P_PROM")
if "df_pend" in globals():
    pct_fuerte = float(df_pend.loc[df_pend["clase"].isin(["fuerte 15-30", "muy fuerte >=30"]),
                                   "pct"].sum())
    indicadores["Pendiente fuerte >15° (%)"] = round(pct_fuerte, 1)
else:
    indicadores["Pendiente fuerte >15° (%)"] = "—"

# --- Precipitación (M2) --------------------------------------------------------
indicadores["Precipitación anual (mm)"] = _num("total_anual", lambda v: v)
indicadores["Lluvia en 3 meses (%)"] = _num("concentracion")
indicadores["Nº meses secos (<30 mm)"] = (
    len(meses_secos) if "meses_secos" in globals() else "—"
)
indicadores["Años secos P33 (frecuencia)"] = (
    int(df_precip["seco"].sum()) if "df_precip" in globals() else "—"
)

# --- Temperatura y heladas (M3, M4) -------------------------------------------
indicadores["Tendencia térmica (°C/década)"] = _num("pendiente_decada")
if "anual_t" in globals():
    indicadores["T mínima media anual (°C)"] = round(float(anual_t["tmin"].mean()), 1)
else:
    indicadores["T mínima media anual (°C)"] = "—"
indicadores["Heladas (días/año)"] = _num("media_h")
indicadores["Años con helada (%)"] = _num("pct_anios")
if "mes_pico" in globals():
    indicadores["Mes pico de helada"] = MESES_ES[int(mes_pico) - 1]
else:
    indicadores["Mes pico de helada"] = "—"

# --- Sequía (M5) y humedad de suelo (M6) --------------------------------------
indicadores["SPI-3 mínimo histórico"] = _num("peor_spi", lambda v: v)
if "por_anio" in globals():
    indicadores["Años con sequía severa (%)"] = round(
        100 * float(por_anio["anio_sequía"].mean()), 1)
else:
    indicadores["Años con sequía severa (%)"] = "—"
if "rachas" in globals():
    indicadores["Máx. racha meses SPI severo"] = int(max(rachas))
else:
    indicadores["Máx. racha meses SPI severo"] = "—"
indicadores["Humedad radicular media (%)"] = _num("serie_sm", lambda v: float(v.mean()))
indicadores["Meses con humedad < P10 (%)"] = _num(
    "df_sm", lambda v: 100 * float(v["seco"].mean()))

# --- Vegetación (M7) -----------------------------------------------------------
indicadores["NDVI pico (curva anual)"] = _num("ndvi_pico", lambda v: v)
if "mes_pico_ndvi" in globals():
    indicadores["Mes pico NDVI"] = MESES_ES[int(mes_pico_ndvi) - 1]
else:
    indicadores["Mes pico NDVI"] = "—"
indicadores["Año NDVI más débil"] = _num("peor_anio_n", lambda v: v)

# --- Agua superficial (M8) y manantiales (M9) ---------------------------------
indicadores["Ocurrencia de agua media (%)"] = _num("occ_aoi", lambda v: v)
indicadores["Estacionalidad agua (meses)"] = _num("est_aoi", lambda v: v)
indicadores["Tendencia área con agua (km²/año)"] = _num("tend_agua", lambda v: v)
indicadores["Manantiales candidatos (n)"] = _num("n_cand", lambda v: v)

# --- Bofedales (M10) -----------------------------------------------------------
if "res_m10" in globals():
    indicadores["Bofedales candidatos (km²)"] = round(res_m10["bof_km2"], 2)
    indicadores["Bofedales (% del AOI)"] = round(res_m10["bof_pct"], 2)
else:
    indicadores["Bofedales candidatos (km²)"] = "—"
    indicadores["Bofedales (% del AOI)"] = "—"

# --- Superficie agrícola en el tiempo (M11) -----------------------------------
if "res_m11" in globals():
    indicadores["Agrícola actual (km²)"] = round(res_m11["agr_km2"], 1)
    indicadores["Agrícola actual (% AOI)"] = round(res_m11["agr_pct"], 1)
    indicadores["Δ agrícola vs. inicio (km²)"] = round(res_m11["agr_delta_km2"], 1)
    indicadores["Δ agrícola vs. inicio (%)"] = round(res_m11["agr_delta_pct"], 1)
    indicadores["Tendencia agrícola (km²/década)"] = round(res_m11["agr_pend_decada"], 1)
    indicadores["Ganancia/pérdida 1ª→últ. (km²)"] = (
        f"{res_m11.get('agr_gan_km2', 0):.1f} / {res_m11.get('agr_perd_km2', 0):.1f}")
else:
    for k in ("Agrícola actual (km²)", "Agrícola actual (% AOI)",
              "Δ agrícola vs. inicio (km²)", "Δ agrícola vs. inicio (%)",
              "Tendencia agrícola (km²/década)", "Ganancia/pérdida 1ª→últ. (km²)"):
        indicadores[k] = "— (ejecutar M11)"

# --- Coberturas (M12) ----------------------------------------------------------
if "res_m12" in globals():
    indicadores["Cobertura dominante (2021)"] = (
        f"{res_m12['cob_top']} ({res_m12['cob_top_pct']:.1f}%)")
    indicadores["Cultivos WorldCover (%)"] = round(res_m12["cob_cultivos_pct"], 1)
else:
    indicadores["Cobertura dominante (2021)"] = "—"
    indicadores["Cultivos WorldCover (%)"] = "—"

# --- Entrega -------------------------------------------------------------------
matriz = pd.DataFrame([indicadores]).T.reset_index()
matriz.columns = ["indicador", ZONA["nombre"]]

print("=" * 70)
print("MATRIZ DE SÍNTESIS TERRITORIAL — una fila por indicador")
print("=" * 70)
display(matriz)

In [ ]:
# ============================================================
# M13.2 Lectura ejecutiva (narrativa automática con los números)
# ============================================================

def _fmt(x, nd=0):
    return f"{x:,.{nd}f}" if isinstance(x, (int, float, np.integer, np.floating)) else str(x)


print("=" * 70)
print(f"LECTURA EJECUTIVA: {ZONA['nombre']} "
      f"({_fmt(area_km2, 1)} km² · {ANIO_INI}–{ANIOS_COMPLETOS[-1]})")
print("=" * 70)

if "total_anual" in globals() and "concentracion" in globals():
    print(f"• CLIMA: {_fmt(total_anual)} mm/año, {concentracion:.0f}% de la lluvia en 3 meses; "
          f"{len(meses_secos)} meses secos. "
          f"Heladas: {_fmt(media_h, 1)} días/año en {pct_anios:.0f}% de los años "
          f"(pico {MESES_ES[int(mes_pico) - 1]}). "
          f"Sequía severa en {100 * por_anio['anio_sequía'].mean():.0f}% de los años "
          f"(SPI mínimo {peor_spi:.1f}).")

if "res_m10" in globals() and "res_m11" in globals():
    print(f"• TERRITORIO PRODUCTIVO: superficie agrícola "
          f"{_fmt(res_m11['agr_km2'], 1)} km² ({res_m11['agr_pct']:.1f}% del AOI) "
          f"con tendencia {res_m11['agr_pend_decada']:+.1f} km²/década "
          f"(Δ {res_m11['agr_delta_km2']:+.1f} km² desde el inicio de la serie); "
          f"bofedales candidatos {_fmt(res_m10['bof_km2'], 2)} km².")

if "occ_aoi" in globals() and "n_cand" in globals():
    print(f"• RECURSO HÍDRICO: ocurrencia de agua {occ_aoi:.1f}% "
          f"({est_aoi:.1f} meses con agua), tendencia del área con agua {tend_agua:+.3f} km²/año, "
          f"{int(n_cand)} candidatos a manantiales por validar.")

if "res_m12" in globals():
    print(f"• COBERTURA: dominante '{res_m12['cob_top']}' "
          f"({res_m12['cob_top_pct']:.1f}%) · cultivos {res_m12['cob_cultivos_pct']:.1f}% (10 m, 2021).")

print("\nPróximos pasos sugeridos: validar manantiales/bofedales en campo, contrastar con")
print("estaciones CLIMBAT y observadores, y repetir la plantilla en zonas comparables.")

## Cierre: checklist para entrar al territorio con evidencia

- [ ] **Parte I ejecutada** (instalación, autenticación, CONFIG con bbox real, utilidades).
- [ ] **M1–M13 ejecutados** sin errores y con la ventana **1996–2026** (o la indicada).
- [ ] **Umbrales revisados** con criterio técnico local (heladas, SPI, bofedales, clases agrícolas).
- [ ] **Puntos de saber local** cargados en `PUNTOS_LOCALES` y contrastados con M9/M10.
- [ ] **Matriz de síntesis (M13)** exportada o copiada para el informe.
- [ ] **Preguntas guía** de cada módulo llevadas a la visita (calendario agrícola vs. fenología).

<details>
<summary><b>💾 Exportar resultados (opcional — descomentar cuando ya esté validado)</b> — clic para ver</summary>
<br>

Este cuaderno no exporta por defecto (fase de pruebas). Para llevar la matriz a Google Drive:

```python
# from google.colab import drive
# drive.mount("/content/drive")
# matriz.to_csv("/content/drive/MyDrive/matriz_sintesis_" +
#               ZONA["nombre"].replace(" ", "_") + ".csv", index=False)
# df_agr.to_csv("/content/drive/MyDrive/superficie_agricola.csv", index=False)
```

**Si aun así aparece `User memory limit exceeded`** (zonas muy grandes o series muy largas):
la plantilla ya reduce en bloques y usa `bestEffort=True`, pero en el peor caso conviene
**exportar la imagen como GeoTIFF** (GEE la calcula por tiles en el servidor, sin traerla a Colab)
y seguir el análisis sobre el archivo. Ejemplo genérico:

```python
# ee.batch.Export.image.toDrive(
#     image=imagen_a_exportar,             # p. ej. n_secos, mapa_heladas
#     description="export_prosuco",
#     region=roi, scale=ESCALAS["clima"], maxPixels=1e10,
# ).start()                                # la tarea aparece en code.earthengine.google.com
```

</details>

### Fuentes de datos y crédito
* Google Earth Engine · Copernicus/ESA (WorldCover, Sentinel) · NASA (MODIS, SMAP, IMERG, NASADEM)
  · USGS (SRTM, Landsat) · ECMWF/Copernicus (ERA5-Land) · UC Santa Barbara (CHIRPS) ·
  JRC/EC (Global Surface Water) · WWF (HydroSHEDS) · University of Maryland (TerraClimate/IDAHOSIN).
* Elaboración: plantilla **PROSUCO · CLIMBAT** — uso educativo y de diagnóstico territorial;
  los resultados satelitales no sustituyen la validación en campo ni constituyen asesoría agronómica.

---

**Fin del cuaderno.** Volver al índice con el panel ☰ de Colab.

## Cierre: checklist del modulo

- [ ] Parte base ejecutada (instalacion, autenticacion GEE, CONFIG, AOI del GeoJSON).
- [ ] Mapa(s) revisados contra el borde rojo del AOI (el GeoJSON siempre manda).
- [ ] Serie/tabla leidas + interpretacion contrastada con saber local (Yapuchiri/PACHASOL).
- [ ] Periodo y dataset anotados junto a cada numero (p. ej. `CHIRPS 1981-2025`).

### Como cambiar de satelite o de anos (patron en todos los modulos)

```python
FECHA_INICIO = "1996-01-01"  # en la celda CONFIG: ventana analitica
FECHA_FIN = "2026-12-31"
# --- Primario (default) ---
precip = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
# --- Alternativas: comentar la primaria y descomentar la elegida ---
# precip = ee.ImageCollection("NASA/GPM_L3/IMERG_V06")
# Cada alternativa indica resolucion, periodo y para que sirve.
```
